# Grocery Sales Forecasting - Model Training and Evaluation
This notebook covers the chronological train/validation splitting, LightGBM model training, metric evaluation, feature importance mapping, and structural error analysis.

## 1. Imports and Feature Loading

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append('../src')
from features import build_features
from validation import get_time_series_split

sns.set_theme(style="whitegrid")

In [ ]:
data_dir = '../data'
train_feat, test_feat = build_features(
    os.path.join(data_dir, 'train.csv'),
    os.path.join(data_dir, 'test.csv'),
    os.path.join(data_dir, 'stores.csv'),
    os.path.join(data_dir, 'oil.csv'),
    os.path.join(data_dir, 'holidays_events.csv')
)

## 2. Chronological Split (Validation Strategy)
We split the dataset into `train_split` and `val_split` using a strict cutoff date (`2017-08-01`). This leaves the final 15 days of the training set for validation, preventing leakage.

In [ ]:
train_split, val_split = get_time_series_split(train_feat)

exclude_cols = ['id', 'date', 'sales', 'log_sales']
feature_cols = [c for c in train_feat.columns if c not in exclude_cols]

X_train = train_split[feature_cols]
y_train = train_split['log_sales']
X_val = val_split[feature_cols]
y_val = val_split['log_sales']

print(f"Training features shape: {X_train.shape}")
print(f"Validation features shape: {X_val.shape}")

## 3. Train LightGBM Model
We train LightGBM Regressor with early stopping on the validation set.

In [ ]:
import lightgbm as lgb

cat_features = [c for c in feature_cols if X_train[c].dtype.name == 'category']
lgb_train = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_features)
lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train, categorical_feature=cat_features)

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 6,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'seed': 42
}

model = lgb.train(
    params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_train, lgb_val],
    callbacks=[lgb.early_stopping(50, verbose=True), lgb.log_evaluation(100)]
)

## 4. Evaluation and Metrics
We evaluate Root Mean Squared Logarithmic Error (RMSLE) on the validation split.

In [ ]:
val_preds = model.predict(X_val, num_iteration=model.best_iteration)
val_preds_clipped = np.clip(val_preds, 0, None)
rmsle = np.sqrt(np.mean((y_val.values - val_preds_clipped) ** 2))
print(f"Validation RMSLE: {rmsle:.5f}")

## 5. Feature Importance

In [ ]:
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

sns.barplot(data=importance.head(15), x='importance', y='feature')
plt.title("Top 15 Feature Importances (Gain)")
plt.show()

## 6. Error Analysis
We segment validation residuals to identify systemic weaknesses in the forecasting architecture.

In [ ]:
val_df_analysis = val_split.copy()
val_df_analysis['pred_log_sales'] = val_preds
val_df_analysis['absolute_log_error'] = np.abs(val_df_analysis['log_sales'] - val_df_analysis['pred_log_sales'])

# Compare error on holidays vs. normal days
holiday_err = val_df_analysis.groupby('is_holiday')['absolute_log_error'].mean()
print("Mean Absolute Log Error:")
print(f"  Holidays:   {holiday_err.get(1, 0.0):.5f}")
print(f"  Normal Days: {holiday_err.get(0, 0.0):.5f}")